# YOLOv8n — Face Detector Training (WIDERFace)

| Item | Detail |
|---|---|
| **Dataset** | WIDERFace for YOLO (Kaggle: lylmsc) |
| **Architecture** | YOLOv8n (nano — fastest variant) |
| **Task** | Single-class object detection: `face` |
| **Framework** | Ultralytics YOLOv8 |
| **Environment** | Kaggle Notebook (GPU T4 / P100) |

---

## Problem Statement

Detect and localise **all human faces** in a video frame, producing bounding boxes that feed into the EfficientNet-B2 emotion classifier (stage 2 of the pipeline).

## Why YOLOv8n?
- **Speed:** The nano variant runs at high FPS on CPU, enabling real-time processing.
- **Single class:** Only one class (`face`) — nano capacity is sufficient; larger variants would overfit.
- **Ultralytics API:** One-line train/export with built-in augmentation, logging, and validation.

---
## 1. Environment Setup

In [ ]:
# Install Ultralytics (includes PyTorch dependency check)
%pip install ultralytics -q

In [ ]:
import os
import random
import yaml
import torch
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")

---
## 2. Configuration

In [ ]:
# ── Dataset paths (Kaggle input) ────────────────────────────────────────────
SRC_IMG_DIR  = '/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training/images'
SRC_LBL_DIR  = '/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training/labels'
OUTPUT_DIR   = '/kaggle/working'
SPLIT_DIR    = os.path.join(OUTPUT_DIR, 'wider_face_split')
DATA_YAML    = os.path.join(OUTPUT_DIR, 'dataset.yaml')

# ── Split ratios ─────────────────────────────────────────────────────────────
TRAIN_RATIO  = 0.80
VAL_RATIO    = 0.10
# Test = remainder (0.10)

# ── Training ─────────────────────────────────────────────────────────────────
MODEL_VARIANT = 'yolov8n.pt'   # nano — best speed/accuracy for single-class face detection
EPOCHS        = 50
IMG_SIZE      = 640
BATCH_SIZE    = 16
DEVICE        = 0 if torch.cuda.is_available() else 'cpu'
NUM_WORKERS   = 2

# ── Detection thresholds ──────────────────────────────────────────────────────
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45

# ── Optimiser ────────────────────────────────────────────────────────────────
OPTIMIZER      = 'AdamW'
LR0            = 1e-3
COSINE_LR      = True          # Cosine annealing schedule
CLOSE_MOSAIC   = 10            # Disable mosaic augmentation in the last N epochs

SEED           = 42
random.seed(SEED)

print(f"Model variant : {MODEL_VARIANT}")
print(f"Epochs        : {EPOCHS}")
print(f"Image size    : {IMG_SIZE}px")
print(f"Device        : {DEVICE}")

---
## 3. Dataset Preparation

**WIDERFace** is a large-scale face detection benchmark with ~12,880 annotated images.
The Kaggle dataset (`lylmsc`) ships annotations in YOLO format (class + normalised xywh per line).

Split strategy:
- **80% Train** — model learning
- **10% Val** — monitored during training for early stopping and mAP reporting
- **10% Test** — held out for final unbiased evaluation

We use **symlinks** instead of copying files to avoid duplicating disk usage on Kaggle.

In [ ]:
# Create split directory structure
for sub in ['images/train', 'images/val', 'images/test',
            'labels/train', 'labels/val', 'labels/test']:
    os.makedirs(os.path.join(SPLIT_DIR, sub), exist_ok=True)

# Collect and shuffle all images with a fixed seed for reproducibility
all_images = sorted([
    f for f in os.listdir(SRC_IMG_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])
random.shuffle(all_images)

total      = len(all_images)
train_end  = int(total * TRAIN_RATIO)
val_end    = train_end + int(total * VAL_RATIO)

splits = {
    'train': all_images[:train_end],
    'val'  : all_images[train_end:val_end],
    'test' : all_images[val_end:],
}

print(f"Total images : {total:,}")
for name, imgs in splits.items():
    print(f"  {name:5s}       : {len(imgs):,}")

In [ ]:
def symlink_split(image_list, split_name):
    """Create symlinks for images and labels into their split subdirectory."""
    for img_name in tqdm(image_list, desc=f"Linking {split_name}"):
        stem    = os.path.splitext(img_name)[0]
        src_img = os.path.join(SRC_IMG_DIR, img_name)
        src_lbl = os.path.join(SRC_LBL_DIR, stem + '.txt')
        dst_img = os.path.join(SPLIT_DIR, 'images', split_name, img_name)
        dst_lbl = os.path.join(SPLIT_DIR, 'labels', split_name, stem + '.txt')

        if os.path.exists(src_img) and not os.path.exists(dst_img):
            os.symlink(src_img, dst_img)
        if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
            os.symlink(src_lbl, dst_lbl)

for split_name, image_list in splits.items():
    symlink_split(image_list, split_name)

print("Dataset split complete.")

In [ ]:
# Write YOLO-format dataset config
data_config = {
    'path' : SPLIT_DIR,
    'train': 'images/train',
    'val'  : 'images/val',
    'test' : 'images/test',
    'nc'   : 1,
    'names': {0: 'face'},
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Dataset YAML written to {DATA_YAML}")
print(yaml.dump(data_config))

---
## 4. Model: YOLOv8n

```
YOLOv8n
├── Backbone  : CSP-Darknet (nano scale)
├── Neck      : PANet feature pyramid
└── Head      : Decoupled detection head
              → 1 class (face)
              → Predict: bbox (x,y,w,h) + objectness + class
```

**Pretrained on COCO** — contains `person` class (includes faces), giving a useful initialisation.

### Training strategy
| Component | Choice | Rationale |
|---|---|---|
| Optimizer | AdamW | Decoupled weight decay; better generalisation than SGD for fine-tuning |
| LR schedule | Cosine annealing | Smooth decay prevents oscillation near convergence |
| Mosaic aug | Disabled last 10 epochs | Stabilises final convergence (close_mosaic=10) |
| cache | False | Avoids Kaggle RAM overflow with 10k+ images |

In [ ]:
if torch.cuda.is_available():
    print(f"GPU ready : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found. Training on CPU will be very slow.")
    print("Enable GPU Accelerator in Kaggle: Settings → Accelerator → GPU T4 x2")

---
## 5. Training

In [ ]:
model = YOLO(MODEL_VARIANT)

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    device      = DEVICE,
    workers     = NUM_WORKERS,
    optimizer   = OPTIMIZER,
    lr0         = LR0,
    cos_lr      = COSINE_LR,
    close_mosaic= CLOSE_MOSAIC,
    conf        = CONF_THRESHOLD,
    iou         = IOU_THRESHOLD,
    save        = True,
    val         = True,
    cache       = False,
    project     = OUTPUT_DIR,
    name        = 'yolov8n_face',
    seed        = SEED,
)

print("Training complete.")

---
## 6. Results & Analysis

In [ ]:
# Ultralytics saves a results.png with all training curves
run_dir    = os.path.join(OUTPUT_DIR, 'yolov8n_face')
results_img = os.path.join(run_dir, 'results.png')

if os.path.exists(results_img):
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(mpimg.imread(results_img))
    ax.axis('off')
    ax.set_title('YOLOv8n Training Curves', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print(f"results.png not found at {results_img}")

In [ ]:
# Ultralytics also generates a confusion matrix
cm_img = os.path.join(run_dir, 'confusion_matrix_normalized.png')
if os.path.exists(cm_img):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(mpimg.imread(cm_img))
    ax.axis('off')
    ax.set_title('Confusion Matrix (normalised)', fontsize=12)
    plt.tight_layout()
    plt.show()

### Validation Metrics (Test Set)

In [ ]:
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
trained_model = YOLO(best_weights)

metrics = trained_model.val(data=DATA_YAML, split='test', conf=0.5, iou=0.5)
print(f"\nTest set metrics")
print(f"  mAP@0.5     : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")

---
## 7. Model Export & Verification

In [ ]:
import shutil

best_src = os.path.join(run_dir, 'weights', 'best.pt')
best_dst = os.path.join(OUTPUT_DIR, 'best.pt')

shutil.copy2(best_src, best_dst)

size_mb = os.path.getsize(best_dst) / (1024 ** 2)
print(f"Exported : {best_dst}")
print(f"Size     : {size_mb:.2f} MB")

# Smoke test
verify = YOLO(best_dst)
import numpy as np
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
out = verify(dummy_frame, verbose=False)
print(f"Smoke test passed: {len(out)} result(s) returned.")
print("Model ready for deployment to services/ai-service/weights/")

---
## 8. Model Card

| Field | Value |
|---|---|
| **Architecture** | YOLOv8n (nano) |
| **Input** | RGB frame, any resolution (internal resize to 640×640) |
| **Output** | Bounding boxes `[x1, y1, x2, y2, conf, class]` |
| **Dataset** | WIDERFace (YOLO format) — 12,880 images |
| **Train split** | 10,304 / 1,288 val / 1,288 test |
| **Classes** | 1 — `face` |
| **Inference conf** | 0.5 |
| **Checkpoint** | `best.pt` (best mAP@0.5 on val) |
| **Framework** | Ultralytics ≥ 8.0 |

### Known limitations
- Small (nano) variant may miss very small faces in crowd scenes.
- Trained on WIDERFace — performance may vary on non-standard lighting or extreme angles.
- Inference speed depends on frame resolution; recommend resizing to 640px long-side before inference.

### Files produced
- `best.pt` — best checkpoint (by val mAP@0.5)
- `yolov8n_face/results.png` — training metrics curves
- `yolov8n_face/confusion_matrix_normalized.png` — confusion matrix
- `dataset.yaml` — YOLO dataset config